In [0]:
# Create separate schemas for each Medallion layer

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Bronze, Silver and Gold schemas are ready!")

# Verify schemas
display(
    spark.sql("""
        SHOW SCHEMAS IN workspace
    """)
)

Bronze, Silver and Gold schemas are ready!


databaseName
bronze
default
ecommerce_lakehouse
gold
information_schema
silver


In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

# Location of our uploaded CSV files
base_path = "/Volumes/workspace/ecommerce_lakehouse/raw_files/"

# Map source CSV files to Bronze table names
datasets = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Process every dataset
for table_name, filename in datasets.items():

    print(f"Processing: {table_name}")

        # Read CSV
    reader = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
    )

    # Handle multiline review comments correctly
    if table_name == "order_reviews":
        reader = (
            reader
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
        )

    df = reader.csv(base_path + filename)

    # Add ingestion metadata
    bronze_df = (
        df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", lit(filename))
    )

    # Write as a Delta table
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bronze.{table_name}")
    )

    print(f"Successfully saved: workspace.bronze.{table_name}")

print("\nBRONZE INGESTION COMPLETED!")

Processing: customers
Successfully saved: workspace.bronze.customers
Processing: geolocation
Successfully saved: workspace.bronze.geolocation
Processing: order_items
Successfully saved: workspace.bronze.order_items
Processing: order_payments
Successfully saved: workspace.bronze.order_payments
Processing: order_reviews
Successfully saved: workspace.bronze.order_reviews
Processing: orders
Successfully saved: workspace.bronze.orders
Processing: products
Successfully saved: workspace.bronze.products
Processing: sellers
Successfully saved: workspace.bronze.sellers
Processing: category_translation
Successfully saved: workspace.bronze.category_translation

BRONZE INGESTION COMPLETED!


In [0]:
bronze_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

results = []

for table_name in bronze_tables:
    df = spark.table(f"workspace.bronze.{table_name}")

    results.append((
        table_name,
        df.count(),
        len(df.columns),
        "_ingested_at" in df.columns,
        "_source_file" in df.columns
    ))

verification_df = spark.createDataFrame(
    results,
    ["table_name", "row_count", "columns",
     "has_ingestion_timestamp", "has_source_file"]
)

display(verification_df)

table_name,row_count,columns,has_ingestion_timestamp,has_source_file
customers,99441,7,true,true
geolocation,1000163,7,true,true
order_items,112650,9,true,true
order_payments,103886,7,true,true
order_reviews,104162,9,true,true
orders,99441,10,true,true
products,32951,11,true,true
sellers,3095,6,true,true
category_translation,71,4,true,true


In [0]:
from pyspark.sql.functions import col

reviews_bronze = spark.table("workspace.bronze.order_reviews")

print("Bronze review records:", reviews_bronze.count())

print(
    "Null review IDs:",
    reviews_bronze.filter(col("review_id").isNull()).count()
)

print(
    "Null order IDs:",
    reviews_bronze.filter(col("order_id").isNull()).count()
)

display(reviews_bronze.limit(5))

Bronze review records: 99224
Null review IDs: 0
Null order IDs: 0


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_ingested_at,_source_file
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,2026-11-03T10:46:29.919Z,olist_order_reviews_dataset.csv
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,2026-11-03T10:46:29.919Z,olist_order_reviews_dataset.csv
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,2026-11-03T10:46:29.919Z,olist_order_reviews_dataset.csv
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,2026-11-03T10:46:29.919Z,olist_order_reviews_dataset.csv
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,2026-11-03T10:46:29.919Z,olist_order_reviews_dataset.csv
